# Research Assistant: LLM Internship Final Project
Run this on **Kaggle** with **GPU T4**. Keep it running while you use the Streamlit app.

In [ ]:
!pip install -q fastapi uvicorn pyngrok "transformers==4.52.4" accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "mistralai/Mistral-Nemo-Instruct-2407"

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4 has no bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_cfg, device_map="auto").eval()

def generate_text(prompt, max_new_tokens=900, temperature=0.1):
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
    kwargs = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    if temperature > 0:
        kwargs.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        kwargs.update(do_sample=False)
    with torch.no_grad():
        out = model.generate(input_ids, **kwargs)
    # decode ONLY the new tokens, so the prompt is not echoed back
    return tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

In [ ]:
print(generate_text("In one sentence, what is retrieval augmented generation?", max_new_tokens=60))

In [ ]:
NGROK_TOKEN = "ADD_NGROK_TOKEN"
API_KEY = "CHANGE"

In [ ]:
import threading
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel

app = FastAPI()
gpu_lock = threading.Lock()   # one generation at a time on the GPU

class GenRequest(BaseModel):
    prompt: str
    max_new_tokens: int = 900
    temperature: float = 0.1

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID}

@app.post("/generate")
def generate(req: GenRequest, authorization: str = Header(default="")):
    if authorization != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    with gpu_lock:
        text = generate_text(req.prompt, req.max_new_tokens, req.temperature)
    return {"response": text}

In [ ]:
import socket, time, uvicorn
from pyngrok import ngrok, conf

def free_port():
    s = socket.socket(); s.bind(("", 0)); port = s.getsockname()[1]; s.close(); return port

port = free_port()
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url
print("Paste this URL in the Streamlit sidebar:", public_url)

threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=port), daemon=True).start()
time.sleep(2)